## Main code of Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class MPM_Node(nn.Module):
    """
    We have implemented a ConvLSTM cell as the main the module of the Motion Pattern Module (MPM).
    """
    def __init__(self, in_channels, hidden_channels, kernel_size=3):
        super(MPM_Node, self).__init__()
        self.padding = kernel_size // 2
        self.conv = nn.Conv2d(
            in_channels + hidden_channels,
            4 * hidden_channels,
            kernel_size, padding=self.padding
        )

    def forward(self, x, h_prev, c_prev):
        combined = torch.cat([x, h_prev], dim=1)
        gates = self.conv(combined)
        i, f, o, g = torch.split(gates, gates.size(1) // 4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)
        c_next = f * c_prev + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next


class MotionVisionAdapter(nn.Module):
    """
    Reduces the semantic gap between vision and motion patterns.
    """
    def __init__(self, channels, T):
        super(MotionVisionAdapter, self).__init__()
        self.T = T
        self.relation_conv = nn.Conv2d(2 * channels, channels // T, kernel_size=1)
        self.adapting_conv = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, h_tilde, h_sequence):
        batch, c, h, w = h_tilde.size()
        outputs = []
        for t in range(self.T):
            h_t = h_sequence[:, :, t, :, :]
            combined = torch.cat([h_tilde, h_t], dim=1)
            relation_weight = F.relu(self.relation_conv(combined))
            adapted = self.adapting_conv(h_t * relation_weight.mean(dim=1, keepdim=True))
            outputs.append(adapted)
        return torch.stack(outputs, dim=2)


class MICPL_Module(nn.Module):
    """
      1. MPM (ConvLSTM) — builds hidden states recurrently over T steps.
      2. MVA — adapts the motion patterns using the last hidden state as reference.
    """
    def __init__(self, channels, T, num_layers: int = 2):
        super(MICPL_Module, self).__init__()
        self.T = T
        self.num_layers = num_layers

        self.mpm_layers = nn.ModuleList([
            MPM_Node(channels, channels) for _ in range(num_layers)
        ])

        self.mva = MotionVisionAdapter(channels, T)

    def forward(self, x_s: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x_s: [B, C, T, H, W] — vision pattern features

        Returns:
            motion_patterns: [B, C, T, H, W]
        """
        batch, channels, T, height, width = x_s.size()

        h = [torch.zeros(batch, channels, height, width, device=x_s.device)
             for _ in range(self.num_layers)]
        c = [torch.zeros(batch, channels, height, width, device=x_s.device)
             for _ in range(self.num_layers)]

        layer_outputs = []

        for t in range(T):
            current_input = x_s[:, :, t, :, :]
            for l in range(self.num_layers):
                h[l], c[l] = self.mpm_layers[l](current_input, h[l], c[l])
                current_input = h[l]
            layer_outputs.append(h[-1])


        h_s = torch.stack(layer_outputs, dim=2)

        h_tilde = h_s[:, :, -1, :, :]  
        motion_patterns = self.mva(h_tilde, h_s)

        return motion_patterns


class SmallObjectDetector(nn.Module):
    def __init__(self, backbone, channels=64, T=5):
        super(SmallObjectDetector, self).__init__()
        self.backbone = backbone
        self.micpl = MICPL_Module(channels, T)

    def forward(self, x_sequence):
        vision_features = []
        for t in range(x_sequence.size(2)):
            feat = self.backbone(x_sequence[:, :, t, :, :])
            vision_features.append(feat)

        x_s = torch.stack(vision_features, dim=2)  

        if self.training:
            h_s = self.micpl(x_s)
            fused_features = x_s + h_s
            return fused_features
        else:
            return x_s

In [ ]:

import torch
import torch.nn as nn
import timm
from torch.hub import load_state_dict_from_url



class BiFPNLayer(nn.Module):
    def __init__(self, channels, eps=1e-4):
        super().__init__()
        self.eps = eps


        self.w_td = nn.Parameter(torch.ones(2, dtype=torch.float32))
        self.td_conv = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1,
                      groups=channels, bias=False),   
            nn.Conv2d(channels, channels, kernel_size=1, bias=False),  
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
        )
        self.w_bu = nn.Parameter(torch.ones(3, dtype=torch.float32))
        self.bu_conv = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1,
                      groups=channels, bias=False),
            nn.Conv2d(channels, channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, p_low, p_mid, p_high):
        """
        p_low  : finest scale  — largest spatial size  (e.g. stride-4)
        p_mid  : mid scale                              (e.g. stride-8)
        p_high : coarsest scale — smallest spatial size (e.g. stride-16)
        Returns a single fused map at p_low's resolution.
        """
        #Top-down: upsample p_high to p_mid size 
        w_td = torch.relu(self.w_td) / (torch.relu(self.w_td).sum() + self.eps)
        p_mid_td = self.td_conv(
            w_td[0] * p_mid +
            w_td[1] * F.interpolate(p_high, size=p_mid.shape[-2:],
                                    mode='bilinear', align_corners=False)
        )

        # Bottom-up: upsample everything to p_low size
        w_bu = torch.relu(self.w_bu) / (torch.relu(self.w_bu).sum() + self.eps)
        p_out = self.bu_conv(
            w_bu[0] * p_low +
            w_bu[1] * F.interpolate(p_mid_td, size=p_low.shape[-2:],
                                    mode='bilinear', align_corners=False) +
            w_bu[2] * F.interpolate(p_high,   size=p_low.shape[-2:],
                                    mode='bilinear', align_corners=False)
        )
        return p_out

class DLA34FeatureExtractor(nn.Module):
    def __init__(self, out_channels=64, pretrained=True,
                 freeze_backbone=True, num_bifpn=2):
        super().__init__()

        self.dla = timm.create_model(
            "hf_hub:timm/dla34.in1k",
            pretrained=pretrained,
            features_only=True,
            out_indices=(1, 2, 3),
        )

        if freeze_backbone:
            self.dla.requires_grad_(False)
            print("DLA-34 backbone frozen.")

        channels = self.dla.feature_info.channels()
        c1, c2, c3 = channels 
        self.proj_c1 = nn.Conv2d(c1, out_channels, kernel_size=1, bias=False)
        self.proj_c2 = nn.Conv2d(c2, out_channels, kernel_size=1, bias=False)
        self.proj_c3 = nn.Conv2d(c3, out_channels, kernel_size=1, bias=False)

        self.bifpn = nn.ModuleList(
            [BiFPNLayer(out_channels) for _ in range(num_bifpn)]
        )

    def unfreeze_backbone(self):
        self.dla.requires_grad_(True)
        print("Backbone unfrozen.")

    def forward(self, x):
        with torch.no_grad():
            f1, f2, f3 = self.dla(x)   
        p1 = self.proj_c1(f1)                        
        p2 = self.proj_c2(f2)                      
        p3 = self.proj_c3(f3)                        


        p1 = F.avg_pool2d(p1, kernel_size=2)         
        for bifpn_layer in self.bifpn:
            p1 = bifpn_layer(p1, p2, p3)             

        return p1   

class SmallObjectDetector(nn.Module):
    def __init__(self, backbone, channels=64, T=5):
        super().__init__()
        self.backbone = backbone
        self.micpl = MICPL_Module(channels, T)

    def forward(self, x_sequence, training=True):

        vision_features = []
        for t in range(x_sequence.size(2)):
            feat = self.backbone(x_sequence[:, :, t, :, :])
            vision_features.append(feat)

        x_s = torch.stack(vision_features, dim=2)  

        if training:
            h_s = self.micpl(x_s)
            return x_s + h_s
        else:
            return x_s


## Dataset Code

In [ ]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from pycocotools.coco import COCO
import albumentations as A
from albumentations.pytorch import ToTensorV2


class VISOSequenceDataset(Dataset):
    def __init__(self, img_dir, ann_file, sequence_length=5, transform=None):

        self.img_dir = img_dir
        self.coco = COCO(ann_file)
        self.sequence_length = sequence_length
        self.transform = transform

        self.img_ids = sorted(self.coco.getImgIds())
        self.valid_starts = len(self.img_ids) - self.sequence_length + 1

    def __len__(self):
        return self.valid_starts

    def _load_image(self, img_id):
        """Load an image by COCO image id, return as RGB numpy array."""
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.img_dir, img_info['file_name'])
        image = cv2.imread(img_path)
        if image is None:
            h = img_info.get('height', 512)
            w = img_info.get('width', 512)
            return np.zeros((h, w, 3), dtype=np.uint8)
        return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    def _get_bboxes(self, img_id):
        """Return valid pascal_voc bboxes and category ids for an image."""
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns    = self.coco.loadAnns(ann_ids)
        bboxes, cat_ids = [], []
        for ann in anns:
            x, y, w, h = ann['bbox']
            if w <= 1 or h <= 1:
                continue
            x2, y2 = x + w, y + h
            if x2 <= x or y2 <= y:
                continue
            bboxes.append([x, y, x2, y2])
            cat_ids.append(ann['category_id'])
        return bboxes, cat_ids

    def __getitem__(self, idx):
        sequence_img_ids = self.img_ids[idx : idx + self.sequence_length]
        target_img_id    = sequence_img_ids[-1]   
        frames = [self._load_image(img_id) for img_id in sequence_img_ids]

        target_bboxes, target_cat_ids = self._get_bboxes(target_img_id)

        if self.transform:
            transformed   = self.transform(
                image=frames[-1],
                bboxes=target_bboxes,
                category_ids=target_cat_ids
            )
            replay_params = transformed['replay']

            
            new_frames = []
            for i in range(self.sequence_length - 1):
                replayed = A.ReplayCompose.replay(
                    replay_params,
                    image=frames[i],
                    bboxes=[],           
                    category_ids=[]
                )
                new_frames.append(replayed['image'])

            new_frames.append(transformed['image'])
            frames     = new_frames
            bboxes     = list(transformed['bboxes'])
            cat_ids    = list(transformed['category_ids'])

        else:
            
            frames  = [
                torch.from_numpy(f.transpose(2, 0, 1)).float() / 255.0
                for f in frames
            ]
            bboxes  = target_bboxes
            cat_ids = target_cat_ids

        
        sequence_tensor = torch.stack(frames, dim=1)

        target = {
            "bboxes": torch.tensor(bboxes,  dtype=torch.float32)
                      if bboxes else torch.zeros((0, 4), dtype=torch.float32),
            "labels": torch.tensor(cat_ids, dtype=torch.int64)
                      if cat_ids else torch.zeros((0,),  dtype=torch.int64),
        }

        return sequence_tensor, target


def collate_fn(batch):
    sequences, targets = zip(*batch)
    return torch.stack(sequences, dim=0), list(targets)



transform = A.ReplayCompose([
    A.Resize(height=512, width=512),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
], bbox_params=A.BboxParams(
    format='pascal_voc',
    label_fields=['category_ids'],
    clip=True,
    min_visibility=0.1,
    min_width=1.0,
    min_height=1.0,
))

IMG_DIR  = './VISO/coco/plane/train2017'
ANN_FILE = './VISO/coco/plane/Annotations/instances_train2017.json'

viso_dataset = VISOSequenceDataset(
    img_dir=IMG_DIR,
    ann_file=ANN_FILE,
    sequence_length=5,
    transform=transform,
)

data_loader = DataLoader(
    viso_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=4,
    collate_fn=collate_fn,
)

In [ ]:
import torch
import numpy as np
import cv2
from torch.utils.data import Dataset, DataLoader
import math

def gaussian_radius(det_size, min_overlap=0.3):
    height, width = det_size

    a1  = 1
    b1  = (height + width)
    c1  = width * height * (1 - min_overlap) / (1 + min_overlap)
    val1 = float(b1 ** 2 - 4 * a1 * c1)
    sq1 = np.sqrt(np.maximum(val1, 0))
    r1  = (b1 + sq1) / 2

    a2  = 4
    b2  = 2 * (height + width)
    c2  = (1 - min_overlap) * width * height
    val2 = float(b2 ** 2 - 4 * a2 * c2)
    sq2 = np.sqrt(np.maximum(val2, 0))
    r2  = (b2 + sq2) / 2

    a3  = 4 * min_overlap
    b3  = -2 * min_overlap * (height + width)
    c3  = (min_overlap - 1) * width * height
        
    val3 = float(b3 ** 2 - 4 * a3 * c3)
    sq3 = np.sqrt(np.maximum(val3, 0))
    r3  = (b3 + sq3) / 2
    
    return min(r1, r2, r3)
class VISOCenterNetDataset(VISOSequenceDataset):
    def __init__(self, img_dir, ann_file, sequence_length=5, transform=None, output_res=128):
        super().__init__(img_dir, ann_file, sequence_length, transform)
        self.output_res = output_res
        self.num_classes = 1

    def draw_gaussian(self, heatmap, center, radius, k=1):
        """Draws a 2D Gaussian kernel on the heatmaps"""
        diameter = 2 * radius + 1
        gaussian = self.gaussian2D((diameter, diameter), sigma=diameter / 6)
        
        x, y = int(center[0]), int(center[1])
        height, width = heatmap.shape[0:2]
        
        left, right = min(x, radius), min(width - x - 1, radius)
        top, bottom = min(y, radius), min(height - y - 1, radius)

        masked_heatmap = heatmap[y - top:y + bottom + 1, x - left:x + right + 1]
        masked_gaussian = gaussian[radius - top:radius + bottom + 1, radius - left:radius + right + 1]
        if min(masked_gaussian.shape) > 0 and min(masked_heatmap.shape) > 0:
            np.maximum(masked_heatmap, masked_gaussian * k, out=masked_heatmap)
        return heatmap

    def gaussian2D(self, shape, sigma=1):
        m, n = [(ss - 1.) / 2. for ss in shape]
        y, x = np.ogrid[-m:m + 1, -n:n + 1]
        h = np.exp(-(x * x + y * y) / (2 * sigma * sigma))
        h[h < np.finfo(h.dtype).eps * h.max()] = 0
        return h
    def __getitem__(self, idx):
        sequence_tensor, raw_target = super().__getitem__(idx)
        bboxes = raw_target["bboxes"]
        
        hm = np.zeros((self.num_classes, self.output_res, self.output_res), dtype=np.float32)
        wh = np.zeros((2, self.output_res, self.output_res), dtype=np.float32)
        reg = np.zeros((2, self.output_res, self.output_res), dtype=np.float32)
        reg_mask = np.zeros((1, self.output_res, self.output_res), dtype=np.float32)

        input_res = 512 
        scale = input_res / self.output_res

        for i in range(len(bboxes)):
            bbox = bboxes[i]
        
            x1 = np.clip(bbox[0], 0, input_res - 1)
            y1 = np.clip(bbox[1], 0, input_res - 1)
            x2 = np.clip(bbox[2], 0, input_res - 1)
            y2 = np.clip(bbox[3], 0, input_res - 1)

            w, h = (x2 - x1) / scale, (y2 - y1) / scale
            
            if h > 0.1 and w > 0.1:
                ct = np.array([(x1 + x2) / (2 * scale), (y1 + y2) / (2 * scale)], dtype=np.float32)
                ct_int = ct.astype(np.int32)
                if ct_int[0] < self.output_res and ct_int[1] < self.output_res:

                    radius = gaussian_radius((h, w), min_overlap=0.7)
                    radius = max(1, int(radius)) 
                    
                    self.draw_gaussian(hm[0], ct_int, radius)
                    wh[0, ct_int[1], ct_int[0]] = w
                    wh[1, ct_int[1], ct_int[0]] = h
                    
                    reg[0, ct_int[1], ct_int[0]] = ct[0] - ct_int[0]
                    reg[1, ct_int[1], ct_int[0]] = ct[1] - ct_int[1]
                    
                    reg_mask[0, ct_int[1], ct_int[0]] = 1

        target = {
            "hm": torch.from_numpy(hm),
            "wh": torch.from_numpy(wh),
            "reg": torch.from_numpy(reg),
            "reg_mask": torch.from_numpy(reg_mask),
            "raw_bboxes": bboxes 
        }

        return sequence_tensor, target

def collate_fn_centrenet(batch):
    sequences = torch.stack([item[0] for item in batch], dim=0)
    targets = {
        "hm": torch.stack([item[1]["hm"] for item in batch], dim=0),
        "wh": torch.stack([item[1]["wh"] for item in batch], dim=0),
        "reg": torch.stack([item[1]["reg"] for item in batch], dim=0),
        "reg_mask": torch.stack([item[1]["reg_mask"] for item in batch], dim=0),
        "raw_bboxes": [item[1]["raw_bboxes"] for item in batch]
    }
    return sequences, targets

viso_dataset = VISOCenterNetDataset(
    img_dir=IMG_DIR,
    ann_file=ANN_FILE,
    sequence_length=5,
    transform=transform,
    output_res=128 # 512 / 4
)

data_loader = DataLoader(
    viso_dataset, 
    batch_size=2, 
    shuffle=True, 
    num_workers=0, 
    collate_fn=collate_fn_centrenet
)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math

class CenterNetHead(nn.Module):
    def __init__(self, in_channels, num_classes=1):
        super(CenterNetHead, self).__init__()
        self.cls_head = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, num_classes, kernel_size=1)
        )
        self.wh_head = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, 2, kernel_size=1)
        )
        self.reg_head = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, 2, kernel_size=1)
        )
        self.cls_head[-1].bias.data.fill_(-2.19)

    def forward(self, x):
        hm = torch.sigmoid(self.cls_head(x)) 
        wh = self.wh_head(x)
        reg = self.reg_head(x)
        return {'hm': hm, 'wh': wh, 'reg': reg}

def focal_loss(pred, gt):
    pos_inds = gt.eq(1).float()
    neg_inds = gt.lt(1).float()
    
    neg_weights = torch.pow(1 - gt, 6)
    
    loss = 0
    pred = torch.clamp(pred, 1e-4, 1 - 1e-4) # Prevent log(0)
    
    pos_loss = torch.log(pred) * torch.pow(1 - pred, 2) * pos_inds
    neg_loss = torch.log(1 - pred) * torch.pow(pred, 2) * neg_weights * neg_inds
    
    num_pos = pos_inds.float().sum()
    pos_loss = pos_loss.sum()
    neg_loss = neg_loss.sum()
    
    if num_pos == 0:
        loss = loss - neg_loss
    else:
        loss = loss - (pos_loss + neg_loss) / num_pos
    return loss

class MICPLLoss(nn.Module):
    def __init__(self, lambda_size=0.1, lambda_off=1.0):
        super(MICPLLoss, self).__init__()
        self.lambda_size = lambda_size
        self.lambda_off = lambda_off

    def forward(self, preds, targets):
        hm_pred = preds['hm']
        wh_pred = preds['wh']
        reg_pred = preds['reg']
        
        hm_gt, wh_gt, reg_gt, reg_mask = targets['hm'], targets['wh'], targets['reg'], targets['reg_mask']
        l_key = focal_loss(hm_pred, hm_gt)
        
        l_size = F.l1_loss(wh_pred * reg_mask, wh_gt * reg_mask, reduction='sum') / (reg_mask.sum() + 1e-4)
    
        l_off = F.l1_loss(reg_pred * reg_mask, reg_gt * reg_mask, reduction='sum') / (reg_mask.sum() + 1e-4)
        
        total_loss = l_key + (self.lambda_size * l_size) + (self.lambda_off * l_off)
        return total_loss, l_key, l_size, l_off

### Pipeline

In [ ]:
from torchvision.ops import nms

def decode_predictions(hm, wh, reg, K=100, nms_threshold=0.3):
    batch, cat, height, width = hm.size()
    
    hmax = F.max_pool2d(hm, kernel_size=3, stride=1, padding=1)
    keep = (hmax == hm).float()
    hm = hm * keep
    
    hm_flat = hm.view(batch, -1)
    scores, inds = torch.topk(hm_flat, K)
    
    ys = (inds // width).float()
    xs = (inds % width).float()
    
    reg = reg.view(batch, 2, -1)
    xs = xs + reg[:, 0, :].gather(1, inds)
    ys = ys + reg[:, 1, :].gather(1, inds)
    
    wh = wh.view(batch, 2, -1)
    ws = wh[:, 0, :].gather(1, inds)
    hs = wh[:, 1, :].gather(1, inds)
    
    bboxes = torch.stack([
        xs - ws / 2, ys - hs / 2,
        xs + ws / 2, ys + hs / 2
    ], dim=-1)
    
    out_boxes, out_scores = [], []
    for i in range(batch):
        keep_idx = nms(bboxes[i], scores[i], iou_threshold=nms_threshold)
        out_boxes.append(bboxes[i][keep_idx])
        out_scores.append(scores[i][keep_idx])
    
    return out_boxes, out_scores  

def calculate_iou(box1, box2):
    """Calculates IoU between two bounding boxes. Handles Tensors and Numpy."""
    if torch.is_tensor(box1): box1 = box1.detach().cpu().numpy()
    if torch.is_tensor(box2): box2 = box2.detach().cpu().numpy()
    
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    inter_area = max(0, x2 - x1) * max(0, y2 - y1)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    return inter_area / (box1_area + box2_area - inter_area + 1e-6)

def calculate_metrics(pred_boxes, pred_scores, gt_boxes, iou_threshold=0.5, score_threshold=0.1):
    tp, fp, fn = 0, 0, 0
    
    # Filter predictions by confidence
    mask = pred_scores > score_threshold
    valid_preds = pred_boxes[mask]
    
    if len(valid_preds) == 0:
        return 0, 0, len(gt_boxes)
    
    matched_gt = set()
    for pred in valid_preds:
        best_iou = 0
        best_gt_idx = -1
        
        for idx, gt in enumerate(gt_boxes):
            if idx in matched_gt: continue
            iou = calculate_iou(pred, gt)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = idx
                
        if best_iou >= iou_threshold:
            tp += 1
            matched_gt.add(best_gt_idx)
        else:
            fp += 1
            
    fn = len(gt_boxes) - len(matched_gt)
    return tp, fp, fn

In [ ]:
from tqdm import tqdm
import time

def train_epoch(model, head, dataloader, optimizer, criterion, device):
    
    model.train()
    head.train()
    total_loss = 0
    
    for sequences, targets in tqdm(dataloader, desc="Training"):
        sequences = sequences.to(device)
        targets_gpu = {
            k: v.to(device) if isinstance(v, torch.Tensor) else v
            for k, v in targets.items()
        }
        
        optimizer.zero_grad()
        fused_features = model(sequences) 
        
        final_features = fused_features[:, :, -1, :, :] 
        preds = head(final_features)
        
        loss, l_key, l_size, l_off = criterion(preds, targets_gpu)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=35) 
        optimizer.step()
        total_loss += loss.item()
        
    return total_loss / len(dataloader)
@torch.no_grad()
def evaluate(model, head, dataloader, device):
    model.eval()
    head.eval()
    
    total_tp, total_fp, total_fn = 0, 0, 0
    start_time = time.time()
    num_frames = 0
    stride = 4 
     
    for sequences, targets in tqdm(dataloader, desc="Evaluating"):
        sequences = sequences.to(device)
        num_frames += sequences.size(0)
        
        vision_features = model(sequences)
        final_features = vision_features[:, :, -1, :, :]
        preds = head(final_features)
        
        pred_bboxes, pred_scores = decode_predictions(preds['hm'], preds['wh'], preds['reg'])
        
        for i in range(sequences.size(0)):
            pb = pred_bboxes[i].clone()
            pb[:, [0, 2]] *= stride
            pb[:, [1, 3]] *= stride
            
            gt_boxes = targets["raw_bboxes"][i]
            
            tp, fp, fn = calculate_metrics(pb, pred_scores[i], gt_boxes, score_threshold=0.5)
            total_tp += tp
            total_fp += fp
            total_fn += fn

    end_time = time.time()
    fps = num_frames / (end_time - start_time)
    
    precision = total_tp / (total_tp + total_fp + 1e-6)
    recall = total_tp / (total_tp + total_fn + 1e-6)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-6)
    
    print(f"Eval Results -> Pr: {precision:.4f} | Re: {recall:.4f} | F1: {f1:.4f} | FPS: {fps:.1f}")
    return f1, precision, recall

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def visualize_predictions(image_tensor, bboxes, scores, score_threshold=0.3):
    
    img = image_tensor.cpu().numpy().transpose(1, 2, 0)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = std * img + mean
    img = np.clip(img, 0, 1)

    fig, ax = plt.subplots(1, figsize=(10, 10))
    ax.imshow(img)
    
    valid_indices = scores > score_threshold
    valid_boxes = bboxes[valid_indices]
    
    for box in valid_boxes:
        x1, y1, x2, y2 = box.cpu().numpy()
        width = x2 - x1
        height = y2 - y1
        
        
        rect = patches.Rectangle(
            (x1, y1), width, height, 
            linewidth=1.5, edgecolor='blue', facecolor='none'
        )
        ax.add_patch(rect)
        
    plt.axis('off')
    plt.show()


In [ ]:
import torch
import torch.nn as nn
import os

import torchvision.models as models

class SimpleFeatureExtractor(nn.Module):
    def __init__(self, out_channels=64):
        super().__init__()
        resnet = models.resnet18(pretrained=True)
        
        self.net = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
            resnet.layer1
        )
        
    def forward(self, x):
        return self.net(x)

if __name__ == "__main__":
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    NUM_GPUS = torch.cuda.device_count()
    NUM_EPOCHS = 55           
    BATCH_SIZE = 4 * max(NUM_GPUS, 1)            
    LEARNING_RATE = 1.25e-4   
    CHANNELS = 64
    SEQ_LEN = 5               
    NUM_CLASSES = 1           
    
    print(f"Using device: {DEVICE}")

    print(f"Using device: {DEVICE} | GPUs available: {NUM_GPUS}")
    IMG_DIR_val = './VISO/coco/plane/val2017'
    ANN_FILE_val = './VISO/coco/plane/Annotations/instances_val2017.json'
    viso_dataset_val = VISOCenterNetDataset(
        img_dir=IMG_DIR_val,
        ann_file=ANN_FILE_val,
        sequence_length=5,
        transform=transform,
        output_res=128 
    )
    
    
    train_loader = DataLoader(viso_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, collate_fn=collate_fn_centrenet)
    val_loader = DataLoader(viso_dataset_val, batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn_centrenet)

    backbone = DLA34FeatureExtractor(out_channels=CHANNELS, pretrained=True, num_bifpn=1)  # BiFPN rounds
    model = SmallObjectDetector(backbone, channels=CHANNELS, T=SEQ_LEN).to(DEVICE)
    head = CenterNetHead(in_channels=CHANNELS, num_classes=NUM_CLASSES).to(DEVICE)

    if NUM_GPUS > 1:
        print(f"Wrapping model and head in DataParallel across {NUM_GPUS} GPUs")
        model = nn.DataParallel(model)
        head  = nn.DataParallel(head)
        torch.autograd.graph.set_warn_on_accumulate_grad_stream_mismatch(False)

    criterion = MICPLLoss(lambda_size=1.0, lambda_off=1.0)
    
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LEARNING_RATE
    )
    print("DLA-34 backbone params:", sum(p.numel() for p in backbone.parameters()))
    print("Model (DLA-34 + MICPL) params:", sum(p.numel() for p in model.parameters()))

    print("Detection head params:",
          sum(p.numel() for p in head.parameters()))

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, 
        T_max=55,      
        eta_min=1e-6   
    )
    f1_scores,pr_scores,recall_scores=[],[],[]
    
    best_f1 = -1.0
    save_dir = "./saved_models"
    os.makedirs(save_dir, exist_ok=True)

    print("Starting training...")
    for epoch in range(NUM_EPOCHS):
        if epoch == 10:
            model.module.backbone.unfreeze_backbone()  
            optimizer.add_param_group({
                'params': model.module.backbone.dla.parameters(),
                'lr': 1e-5  
            })
        print(f"\n--- Epoch [{epoch+1}/{NUM_EPOCHS}] ---")
        
        avg_train_loss = train_epoch(model, head, train_loader, optimizer, criterion, DEVICE)
        print(f"Average Training Loss: {avg_train_loss:.4f}")
        
        f1, precision, recall = evaluate(model, head, val_loader, DEVICE)
        f1_scores.append(f1)
        pr_scores.append(precision)
        recall_scores.append(recall)
        
        
        scheduler.step() 
        
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Current Learning Rate: {current_lr}")

        if f1 > best_f1:
            best_f1 = f1
            
            save_path = os.path.join(save_dir, "best_micpl_model.pth")
            torch.save({
                'epoch': epoch,
                
                'model_state_dict': model.module.state_dict() if NUM_GPUS > 1 else model.state_dict(),
                'head_state_dict':  head.module.state_dict()  if NUM_GPUS > 1 else head.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_f1': best_f1,
            }, save_path)
            print(f"New best model saved with F1: {best_f1:.4f}")

    print("Training complete! Best F1 Score:", best_f1)

In [ ]:
import matplotlib.pyplot as plt


epochs = range(1, NUM_EPOCHS + 1)

plt.figure(figsize=(10, 6))
plt.plot(epochs, f1_scores,      label='F1',        color='royalblue',  linewidth=2, marker='o', markersize=3)
plt.plot(epochs, pr_scores,      label='Precision',  color='darkorange', linewidth=2, marker='s', markersize=3)
plt.plot(epochs, recall_scores,  label='Recall',     color='seagreen',   linewidth=2, marker='^', markersize=3)


for milestone in [35, 45]:
    plt.axvline(x=milestone, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    plt.text(milestone + 0.3, 0.02, f'LR drop (ep {milestone})', 
             fontsize=8, color='gray', rotation=90, va='bottom')

best_epoch = f1_scores.index(max(f1_scores)) + 1
plt.axvline(x=best_epoch, color='royalblue', linestyle=':', linewidth=1.2, alpha=0.6)
plt.text(best_epoch + 0.3, max(f1_scores) - 0.05,
         f'Best F1: {max(f1_scores):.4f} (ep {best_epoch})',
         fontsize=8, color='royalblue')

plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('F1 / Precision / Recall vs Epochs', fontsize=14)
plt.legend(fontsize=11)
plt.ylim(0, 1.05)
plt.xlim(1, NUM_EPOCHS)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('./saved_models/metrics_plot.png', dpi=150)
plt.show()
print("Plot saved to ./saved_models/metrics_plot.png")

In [ ]:
import matplotlib.pyplot as plt
import cv2

def unnormalize(img):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img * std[:, None, None] + mean[:, None, None]
    return img

def visualize_sample(model, head, val_loader, device):
    model.eval()
    head.eval()

    sequences, targets = next(iter(val_loader))
    sequences = sequences.to(device)

    with torch.no_grad():
        fused = model(sequences)
        final_feat = fused[:, :, -1, :, :]
        preds = head(final_feat)

    bboxes, scores = decode_predictions(
        preds['hm'], preds['wh'], preds['reg'], K=100
    )
    seq = sequences[0].cpu().numpy()
    gt_boxes = targets["raw_bboxes"][0]

    pred_boxes = bboxes[0].cpu().numpy()
    pred_scores = scores[0].cpu().numpy()
    img = seq[:, -1]
    img = unnormalize(img).transpose(1, 2, 0)
    img = np.clip(img, 0, 1)
    img = (img * 255).astype(np.uint8)
    img = cv2.resize(img, (512, 512))

    for box in gt_boxes:
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

    scale = 512 / 128  

    for i in range(len(pred_boxes)):
        score = pred_scores[i]

        if score < 0.3:
            continue

        x1, y1, x2, y2 = pred_boxes[i]

        x1 = int(x1 * scale)
        y1 = int(y1 * scale)
        x2 = int(x2 * scale)
        y2 = int(y2 * scale)

        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(img, f"{score:.2f}", (x1, y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 1)

    plt.figure(figsize=(6,6))
    plt.imshow(img)
    plt.title("Green = GT | Red = Pred")
    plt.axis('off')
    plt.show()

In [ ]:
visualize_sample(model,head,val_loader,DEVICE)

### Testing

In [ ]:
IMG_DIR_test = './VISO/coco/plane/test2017'
ANN_FILE_test = './VISO/coco/plane/Annotations/instances_test2017.json'
viso_dataset_test = VISOCenterNetDataset(
    img_dir=IMG_DIR_test,
    ann_file=ANN_FILE_test,
    sequence_length=5,
    transform=transform,
    output_res=128 
)
test_loader = DataLoader(viso_dataset_test, batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn_centrenet)

In [ ]:
checkpoint = torch.load("./saved_models/best_micpl_model.pth", map_location=DEVICE)
backbone = DLA34FeatureExtractor(out_channels=CHANNELS, pretrained=True, num_bifpn=1)
model = SmallObjectDetector(backbone, channels=CHANNELS, T=SEQ_LEN).to(DEVICE)
head = CenterNetHead(in_channels=CHANNELS, num_classes=NUM_CLASSES).to(DEVICE)

model.load_state_dict(checkpoint['model_state_dict'])
head.load_state_dict(checkpoint['head_state_dict'])

model.eval()
head.eval()

print(f"Loaded model from epoch {checkpoint['epoch']} with best F1: {checkpoint['best_f1']:.4f}")

In [ ]:
f1, precision, recall = evaluate(model, head, test_loader, DEVICE)